<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-12-production-deploy/lesson-12.8-integrate-surfaces/notebooks/GCP_Capstone_12.8_IntegrateSurfaces.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.8 Integrate the Surfaces — One Answer to "Who Is Asking?"
**Netsetos GenAI Engineering — GCP Capstone** · Module 12 · rebuilt on the live lane, 10 September 2026

Four surfaces had four answers to one question; now there is one verifier, one roster, and a smoke that asks for refusals. This notebook asks the verifier four ways on the live API - the member, the outsider, nobody, a forged assertion - and gets four different answers for four different reasons. It asks the chat surface as a member and with a tenant in the body. It reads the roster in both directions. It runs the seven smokes as one and counts the lines that are refusals. And it turns the lane off, reading back every floor and saying plainly what Terraform recreates and what nothing does. The first version's tables stay where they still teach; the six heredocs at the end are the kit's source.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-firestore==2.30.0 fastmcp==3.4.7 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "feat/lesson-4.8-live-evals"        # the demo branch; main is behind it

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
CHAT_URL  = f"https://documind-chat-{NUMBER}.{REGION}.run.app"    # 8.7: the chat service, IAP on it, the API behind it
AGENT_URL = f"https://documind-agent-{NUMBER}.{REGION}.run.app"   # 8.6: the A2A peer
MCP_URL   = f"https://documind-mcp-{NUMBER}.{REGION}.run.app"     # 7.2: the MCP server
QUESTION  = "After how many years of continuous service does gratuity become payable?"

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120, token: str | None = "member") -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) - as documind-ui-sa by default, with token=None as
    nobody, or with a token minted as another account. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    headers = {}
    if token == "member":
        headers["Authorization"] = f"Bearer {documind_tools._id_token(API_URL)}"     # the audience is the canonical URL
    elif token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(f"{url}{path}", json=body, headers=headers, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def api_get(path: str, base: str | None = None, timeout: int = 60) -> tuple[int, dict | str]:
    """GET one API route as the roster member: /version, /health."""
    url = (base or API_URL).rstrip("/")
    r = requests.get(f"{url}{path}", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"), "sa": j["spec"]["template"]["spec"].get("serviceAccountName"),
            "annotations": (j["spec"]["template"]["metadata"].get("annotations") or {}),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query", service_name: str = "documind-api") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="{service_name}" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

# THE TWENTY LINES THAT MATTER, FROM THE CLONE. Module 12's notebooks are where the kit's files come from (the heredoc
# cells at the end of each notebook are what extract_documind.py reads), so the walls stay there and the story reads
# the file the lane actually runs, around one line, with the file's length beside it.
def excerpt(rel: str, needle: str, before: int = 0, after: int = 14) -> str:
    lines = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read().splitlines()
    i = next(n for n, l in enumerate(lines) if needle in l)
    lo, hi = max(0, i - before), min(len(lines), i + after)
    return f"# {rel}:{lo + 1}-{hi}  ({len(lines)} lines)\n" + "\n".join(lines[lo:hi])

def gcloud(*args: str) -> str:
    """One gcloud read, as the notebook's account, stdout only."""
    r = subprocess.run(["gcloud", *args, "--project", PROJECT_ID], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else f"(gcloud: {r.stderr.strip()[:200]})"

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # run_eval, usage_rows, judge
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules, for the excerpts and the pure functions
print("helpers: api(), api_get(), gateway(), service(), usage_rows(), excerpt(), gcloud(); the kit's evals/ and rag-api/ on sys.path")


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def api_as(service_account: str, path: str, body: dict) -> tuple[int, dict | str]:
    """The same route, as a DIFFERENT account - the outsider cells."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {id_token_as(service_account, API_URL)}"}, timeout=120)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]


## Cell 2: Four surfaces, four answers


In [ ]:
# Four surfaces. Four different answers to one question.
#
# DocuMind is finished. 12.7 ships it. So the only thing left is the question every
# one of these services has to answer before it does anything: WHO IS ASKING?
SURFACES = [
    # service,           file,                       how it decides who you are
    ('documind-api',     'services/rag-api/auth.py',
     'verifies the IAP assertion, then checks the Firestore tenant roster'),
    ('documind-admin',   'services/admin/auth.py',
     'verifies the IAP assertion, then an ADMIN_EMAILS allowlist'),
    ('documind-ui',      'services/frontend/auth.py',
     'verifies the IAP assertion with a DIFFERENT library'),
    ('documind-chat',    'services/chat/agent.py',
     'does not decide. It BELIEVES the request body.'),
]
for svc, f, how in SURFACES:
    print(f'  {svc:16} {f:30} {how}')
print()
print('  Three of those are defensible. The fourth is a cross-tenant read with')
print('  no attacker required - just a curl.')
print()
print('  And rag-api/auth.py opens with the sentence that condemns it:')
print('      "The tenant is something you ARE, not something you send."')
print('  The chat service was written after that sentence, by people who had')
print('  read it, and it still takes tenant_id off the request body. That is')
print('  what integration work IS: not building anything new, but making four')
print('  things that were each built correctly agree with each other.')


## Cell 3: One verifier, asked four ways
> A header is a claim; a signature is a credential. Reading `x-goog-authenticated-user-email` is not a shortcut, it is the absence of a check: anything that can reach the service can set a header.


In [ ]:
# ONE VERIFIER, ASKED FOUR WAYS. shared/iap.identity() is the only answer to "who is asking" on every surface: the
# assertion first (a person, through IAP), else the bearer token (an account, through IAM), never a header. Four
# requests to the API prove its four branches: the member answered; the outsider - identified, then refused by the
# roster - 403; nobody - refused by Cloud Run IAM before the app; and a FORGED assertion beside a valid token - 401,
# because the assertion wins and the forgery fails verification. Two refusals from the app, two different reasons.
body = {"query": QUESTION, "tenant_id": TENANT, "user_id": "u_12_8", "top_k": 5, "stream": False}
member = documind_tools._id_token(API_URL)
st_m, ans = api("/v1/query", body)
st_o, out = api("/v1/query", body, token=id_token_as(OUTSIDER_SA, API_URL))
st_n, none = api("/v1/query", body, token=None)
forged = requests.post(f"{API_URL}/v1/query", json=body, timeout=120,
                       headers={"Authorization": f"Bearer {member}", "x-goog-iap-jwt-assertion": "eyJhbGciOiJFUzI1NiJ9.forged.signature"})
print(f"member          : {st_m}  answerable={ans.get('answerable') if isinstance(ans, dict) else ans}")
print(f"outsider        : {st_o}  {str(out)[:80]}   <- identified (via iam), not on the roster")
print(f"nobody          : {st_n}  {'refused by the platform' if st_n in (401, 403) else none}")
print(f"forged assertion: {forged.status_code}  {forged.text[:80]}   <- the assertion wins, and it does not verify")
assert st_m == 200 and st_o == 403 and st_n in (401, 403) and forged.status_code == 401
print()
print(excerpt("shared/iap.py", "def identity(", 0, 18))


### A header is a claim, a signature is a credential


In [ ]:
# A header is a claim. A signature is a credential.
#
# Anything that can reach your service can SET a header. The whole point of
# Identity-Aware Proxy is that it sets one you can CHECK.
REQUESTS = [
    ('curl -H "x-goog-authenticated-user-email: ceo@acme.in"',
     'a header the caller typed', False),
    ('curl -H "x-goog-iap-jwt-assertion: eyJhbGci..."',
     'a JWT signed by Google, IF you verify it', True),
    ('curl -H "x-goog-iap-jwt-assertion: eyJhbGci..."  (unverified)',
     'a base64 string the caller typed', False),
]
print(f'  {"what arrives":58} trustworthy?')
for req, what, ok in REQUESTS:
    print(f'  {req:58} {"yes" if ok else "NO":>4}   {what}')
print()
print('  Row 3 is the one that catches people. Reading the assertion and')
print('  decoding it WITHOUT checking the signature is exactly as safe as row 1,')
print('  and it looks much safer, because the variable is called `claims`.')
print()

# What verification actually has to establish. Miss any one and it is not verification.
CHECKS = [
    ('signature', 'signed by Google IAP',        'the JWKS at .../iap/verify/public_key'),
    ('audience',  'minted for THIS service',     'without it, a token for ANY service passes'),
    ('issuer',    'https://cloud.google.com/iap', 'not some other Google token'),
    ('expiry',    'still valid',                 'the library does this if you let it'),
]
for name, what, why in CHECKS:
    print(f'  {name:10} {what:32} {why}')
print()
print('  AUDIENCE is the one worth stopping on. It is the only check that makes')
print('  the token specific to YOUR service. Verify without it and a token minted')
print('  for the admin dashboard - which a legitimate admin can obtain - is')
print('  accepted by the API. Two of DocuMind\'s three verifiers refuse to start')
print('  when IAP_AUDIENCE is unset. The third does not. Step 3.')


### The audience string, and four ways to get it wrong


In [ ]:
# The string that is a permanent 401 if you get it wrong.
#
# For a Cloud Run service with IAP enabled directly, the audience is:
AUD = '/projects/{PROJECT_NUMBER}/locations/{REGION}/services/{SERVICE}'
print(f'  {AUD}')
print()
MISTAKES = [
    ('projects/123.../services/documind-ui', 'no leading slash'),
    ('/projects/documind-ai-YOUR-ID/...',    'project ID, not project NUMBER'),
    ('https://documind-ui-xyz.run.app',      'the URL - that is the IAM audience, not IAP'),
    ('/projects/123.../services/documind-api', 'the API never mints one; the UI does'),
]
print('  Four ways to get it wrong, all of which produce the same symptom:')
for wrong, why in MISTAKES:
    print(f'    {wrong:42} {why}')
print()
print('  The symptom is a 401 on every request, for everyone, forever - which')
print('  reads as "login is broken" and sends you to look at IAP, the OAuth')
print('  consent screen and the IAM bindings before you look at a string.')
print()
print('  Get the number with:')
print('    gcloud projects describe $PROJECT --format="value(projectNumber)"')
print()
print('  And note the last row. documind-api verifies an assertion IAP minted for')
print('  documind-ui, because the UI is what the browser reached. Its IAP_AUDIENCE')
print('  is therefore the UI\'s audience, not its own. No file in this repo said')
print('  that before today, and the obvious guess is the one that fails.')


### Three copies of one routine, and what one verifier has to get right


In [ ]:
# Three copies of one security routine, in three shapes.
#
# Nobody decided this. It happened because 12.2, 12.3 and 12.4 were written in
# that order, and each one solved the problem again.
IMPL = [
    ('rag-api',  'google.oauth2.id_token.verify_token', 'verify/public_key',
     'raise 500 if IAP_AUDIENCE unset'),
    ('admin',    'google.oauth2.id_token.verify_token', 'verify/public_key',
     'st.stop() if IAP_AUDIENCE unset'),
    ('frontend', 'PyJWT PyJWKClient + jwt.decode',      'verify/public_key-jwk',
     'passes audience=None straight to the library'),
]
print(f'  {"service":10} {"library":38} {"key source":22} missing audience')
for svc, lib, keys, missing in IMPL:
    print(f'  {svc:10} {lib:38} {keys:22} {missing}')
print()
print('  Two libraries, two key endpoints, three behaviours when the audience is')
print('  missing. None of them is WRONG on its own. The problem is that there are')
print('  three, because now a fix has to be made three times and will be made once.')
print()
print('  This is the shared/pii.py argument from deploy/UNOWNED.md, which the kit')
print('  already makes about a different list:')
print('      "Two lists that drift is the worst kind of compliance bug: the scan')
print('       misses a type, the dashboard reports zero findings, and both look')
print('       correct."')
print('  Swap "list" for "verifier" and it is this slide.')
print()
print('  So 12.8 writes ONE shared/iap.py and has all four surfaces import it -')
print('  the same move 8.7 made for retrieve(), enforced by')
print('  tools/check_one_retrieval.py. A rule nothing checks is a preference.')


In [ ]:
# Four surfaces, one verifier. What shared/iap.py had to get right.
#
# Not a refactor for tidiness. Each of the three implementations had a different
# answer to "what happens when IAP_AUDIENCE is unset", and one of them was wrong.
BEHAVIOUR = [
    ('rag-api',  'raise 500', 'fails closed'),
    ('admin',    'st.stop()', 'fails closed'),
    ('frontend', 'pass audience=None to the library',
     'whatever the library does - undecided'),
    ('chat',     'no verification at all', 'fails OPEN'),
]
for svc, what, verdict in BEHAVIOUR:
    print(f'  {svc:10} {what:36} {verdict}')
print()
print('  Row 3 is the interesting failure. It is not that the answer is wrong -')
print('  it is that nobody chose it. The security property was whatever a')
print('  dependency happened to do with None, which is a property that can change')
print('  in a patch release you did not read.')
print()
print('  shared/iap.py decides, once:')
print()
RULES = [
    ('no IAP_AUDIENCE configured', 'REFUSE. Empty means nobody, never everybody.'),
    ('no assertion header',        'REFUSE - the request did not come through IAP.'),
    ('bad signature / expired',    'REFUSE.'),
    ('issuer is not IAP',          'REFUSE.'),
    ('audience not in the list',   'REFUSE.'),
    ('no email claim',             'REFUSE - a verified nobody is still nobody.'),
]
for case, then in RULES:
    print(f'    {case:30} -> {then}')
print()
print('  IAP_AUDIENCE is a LIST, and that is not over-engineering. documind-api is')
print('  not behind IAP - nothing reaches it from a browser - so the assertion it')
print('  sees was minted for whichever SURFACE forwarded it, and there are two.')
print('  A service that accepts forwarded assertions has to name every audience it')
print('  will accept, and refuse the rest.')


### The empty string that made everybody an admin


In [ ]:
# The empty string that made everybody an admin.
#
# services/frontend/auth.py, module level. This is real, it is in the repo, and
# it is one character of difference from the file next door.
import os

os.environ.pop('ADMIN_EMAILS', None)          # the deploy command for documind-ui
os.environ.pop('ADMIN_DOMAINS', None)         # sets neither of these

# --- frontend/auth.py, verbatim
ADMIN_EMAILS = set(os.getenv('ADMIN_EMAILS', '').split(','))
ADMIN_DOMAINS = set(os.getenv('ADMIN_DOMAINS', '').split(','))


def is_admin(user):
    email = user.get('email', '').lower()
    domain = email.split('@')[-1] if '@' in email else ''
    return email in ADMIN_EMAILS or domain in ADMIN_DOMAINS


print(f'  ADMIN_EMAILS  parses to {ADMIN_EMAILS!r}')
print(f'  ADMIN_DOMAINS parses to {ADMIN_DOMAINS!r}')
print()
for label, user in [('a normal user', {'email': 'priya@acme.in'}),
                    ('a user whose email is empty', {'email': ''}),
                    ('a user dict with no email at all', {})]:
    print(f'  {"ADMIN" if is_admin(user) else "not admin":10}  {label}')
print()
print("  ''.split(',') is [''], not []. So the allowlist is not empty - it")
print("  contains the empty string, and any user whose email is missing matches")
print('  it. An unset allowlist should mean "nobody". Here it means "anybody')
print('  without an email".')
print()

# --- admin/auth.py parses the SAME env var, one clause longer
ADMIN_OK = [e.strip().lower() for e in os.environ.get('ADMIN_EMAILS', '').split(',') if e.strip()]
print(f'  admin/auth.py parses the same variable to {ADMIN_OK!r}')
print('  The whole difference is `if e.strip()`.')
print()
print('  Two files, same env var, same intent, different result. That is what')
print('  step 3 means by drift, and it is why the fix is one module rather than')
print('  three careful reviews.')


## Cell 4: IAP and IAM answer different questions
Both legs travel on the same request to the API: the bearer token says the caller may call, the assertion says who it is calling for.


In [ ]:
# IAP and IAM answer different questions. Using one for the other is the
# most common mistake in this whole module.
HOPS = [
    ('a person -> documind-ui',    'IAP', 'a browser, a Google login, a signed assertion'),
    ('a person -> documind-admin', 'IAP', 'same, plus an admin allowlist'),
    ('a person -> documind-chat',  'IAP', 'this lesson adds it'),
    ('documind-ui -> documind-api', 'IAM', 'no browser exists; there is nobody to log in'),
    ('documind-chat -> documind-api', 'IAM', 'same'),
    ('Eventarc -> documind-ingest', 'IAM', 'a service, invoked by a service'),
]
print(f'  {"hop":34} {"mechanism":10} why')
for hop, mech, why in HOPS:
    print(f'  {hop:34} {mech:10} {why}')
print()
print('  The rule is short: IAP is for a HUMAN in a BROWSER. IAM is for one')
print('  service calling another. IAP cannot do the second because there is no')
print('  browser to redirect and nobody to sign in.')
print()

# What each hop actually carries.
def credential_for(hop_kind: str) -> str:
    if hop_kind == 'human':
        return ('IAP intercepts the request, makes the person sign in, and adds\n'
                '      x-goog-iap-jwt-assertion, signed by Google, audience = this service.')
    return ('The CALLING service mints an ID token for the target URL:\n'
            '      documind_tools._id_token(target_url)   # shared/documind_tools.py, lesson 7.3\n'
            '      and sends it as Authorization: Bearer. Cloud Run checks\n'
            '      roles/run.invoker before your code runs at all.')


for kind in ('human', 'service'):
    print(f'  {kind.upper()}:')
    print(f'      {credential_for(kind)}')
    print()
print('  documind-api is deployed --no-allow-unauthenticated and WITHOUT --iap,')
print('  and that is correct, not an oversight: nothing reaches it from a browser.')
print('  Its callers are documind-ui and documind-chat, both of which hold')
print('  roles/run.invoker on it. shared/documind_tools.py already mints that')
print('  token per call - lesson 7.3 built it.')


In [ ]:
# The other leg. IAP cannot do this one, and that is not a limitation.
#
# documind-ui -> documind-api has no browser in it. There is nobody to redirect
# to a Google login, so there is no assertion for IAP to mint.
def credential_for(hop: str) -> tuple[str, str]:
    if hop == 'person -> surface':
        return ('IAP',
                'IAP intercepts, makes the person sign in, and adds a signed\n'
                '           x-goog-iap-jwt-assertion whose audience is that service.')
    return ('IAM',
            'The CALLING service mints an ID token for the target URL and sends\n'
            '           it as Authorization: Bearer. Cloud Run checks\n'
            '           roles/run.invoker BEFORE your code runs at all.')


for hop in ('person -> surface', 'service -> service'):
    mech, how = credential_for(hop)
    print(f'  {hop:20} {mech}')
    print(f'           {how}')
    print()
print('  Both legs travel on the SAME request to documind-api:')
print('    Authorization: Bearer <ID token>       <- IAM: may this service call me?')
print('    x-goog-iap-jwt-assertion: <JWT>        <- IAP: who is the human?')
print()
print('  Cloud Run checks the first before your code runs. Your code checks the')
print('  second. Neither is optional and neither substitutes for the other: the')
print('  first says the CALLER is allowed, the second says WHO it is calling for.')
print()
print('  Which is why shared/documind_tools.py was wrong in a specific way. It')
print('  minted the ID token correctly (lesson 7.3) and then invented the second')
print('  half - "x-user-email" and "x-tenant-id" - under a comment saying rag-api')
print('  read them. It did not. Every agent retrieval got a 401 and returned')
print('  {"error": "document retrieval is unavailable"} AS DATA, which the model')
print('  then explained politely. Indistinguishable from an empty corpus.')
print()
print('  FIXED (gap G4, 2026-09-05). The tool forwards the assertion of the person')
print('  the call is for, when there is one, beside its own ID token - and rag-api')
print('  verifies BOTH legs through shared/iap.identity(): the assertion first')
print('  (who is the human?), else the bearer token itself (which account is')
print('  calling? - an agent in a notebook, run_eval.py, smoke.py). The invented')
print('  headers are gone from every lesson that sent them: 7.1, 8.1-8.5, 8.7.')


## Cell 5: The surface that had no identity
> `tenant_id` and `user_id` used to be fields of the chat request, and the agent believed them. Removing them is stronger than validating them: a validation is something a later edit can loosen; a field that does not exist is not. There is nowhere in this request to lie.


In [ ]:
# THE SURFACE THAT HAD NO IDENTITY. documind-chat took tenant_id in the request body until 12.8: the caller picked the
# tenant. Now ChatRequest has no such field, and the tenant is what the verified caller IS (tenancy.tenant_for).
# Removing the field is stronger than validating it: a validation is something a later edit can loosen, a field that
# does not exist is not. Asked as the member: an answer through the brain the service runs. Asked with a tenant in
# the body: the same answer, and the row the service logs names the CALLER's tenant, whatever the body said.
chat_tok = documind_tools._id_token(CHAT_URL)
r = requests.post(f"{CHAT_URL}/v1/chat", json={"question": QUESTION, "session_id": "l128"}, headers={"Authorization": f"Bearer {chat_tok}"}, timeout=150)
print("member:", r.status_code, {k: (str(v)[:70] if k != "citations" else len(v)) for k, v in (r.json() if r.status_code == 200 else {"body": r.text}).items() if k in ("answer", "brain", "tenant_id", "citations", "body")})
r2 = requests.post(f"{CHAT_URL}/v1/chat", json={"question": QUESTION, "session_id": "l128b", "tenant_id": "zeta"}, headers={"Authorization": f"Bearer {chat_tok}"}, timeout=150)
print("tenant in the body:", r2.status_code, (r2.json().get("answer") or "")[:70] if r2.status_code == 200 else r2.text[:100])
time.sleep(20)
rows = usage_rows(minutes=3, limit=2, event="chat", service_name="documind-chat")
print("the service's rows say tenant:", [row.get("tenant") for row in rows], "- the body said zeta")
assert r.status_code == 200 and r2.status_code == 200, "the field does not exist; the request is valid without it and unchanged with it"
assert rows and all(row.get("tenant") == TENANT for row in rows), "the tenant is something you are: looked up, never received"
print()
print(excerpt("services/chat/agent.py", "class ChatRequest(", 0, 12))


### The tenant is something you are


In [ ]:
# The tenant is something you ARE, not something you send.
#
# services/chat/agent.py, the shipped version:
BEFORE = """class ChatRequest(BaseModel):
    question: str
    tenant_id: str        # <- the CALLER chooses this
    user_id: str          # <- and this

@app.post("/v1/chat")
def chat(req: ChatRequest) -> dict:
    result = agent.invoke(..., context={"tenant_id": req.tenant_id, ...})"""
print(BEFORE)
print()
print('  One curl away from any tenant\'s documents:')
print("      curl -d '{\"question\":\"travel cap?\",\"tenant_id\":\"acme\",")
print("                \"user_id\":\"x\"}' https://documind-chat.../v1/chat")
print()
print('  Note what this defeats. The retriever DOES filter by tenant - correctly,')
print('  with a hard predicate. It filters by the tenant it was handed. A perfect')
print('  filter on an attacker-supplied value is not a filter.')
print()

AFTER = """def identity(request: Request) -> dict:
    \"\"\"Who, verified - from shared/iap.py, the one verifier.\"\"\"
    return verify_iap(request)            # 401 if the assertion is bad

@app.post("/v1/chat")
def chat(req: ChatRequest, user=Depends(identity)) -> dict:
    tenant_id = tenant_for(user["email"])          # Firestore roster lookup
    if tenant_id is None:
        raise HTTPException(403, "not a member of any tenant")
    result = agent.invoke(..., context={"tenant_id": tenant_id,
                                        "user_id": user["email"]})"""
print(AFTER)
print()
print('  tenant_id and user_id leave the request schema entirely. There is no')
print('  field to lie in. That is stronger than validating the field, because a')
print('  validation is something a future edit can loosen and a missing field is')
print('  not.')
print()
print('  The roster lookup is the same one documind-ui already does')
print('  (frontend/auth.py:tenant_for) and the same collection rag-api checks')
print('  (auth.py:is_member). Three surfaces, one roster - which is the point.')


## Cell 6: One roster, both directions


In [ ]:
from shared import tenancy

# ONE ROSTER, BOTH DIRECTIONS. tenants/{tenant}.members is the document the API, the UI and the chat all consult
# (make roster wrote it): is_member(email, tenant) is the API's 403, tenant_for(email) is the chat's tenant. The
# outsider is in no roster - that is what makes it the outsider - and the roster member is in three.
for t in ("acme", "zeta", "globex"):
    members = tenancy.list_members(t)
    print(f"  {t:7} {len(members)} member(s): {[m.split('@')[0] for m in members]}")
print()
for who in (MEMBER_SA, OUTSIDER_SA):
    print(f"  {who.split('@')[0]:22} tenant_for -> {tenancy.tenant_for(who)!r}   is_member(acme) -> {tenancy.is_member(who, TENANT)}")
assert tenancy.is_member(MEMBER_SA, TENANT) and not tenancy.is_member(OUTSIDER_SA, TENANT)
print("\nadd_member / list_members are the only writers and readers; a roster edit lands on every surface at once")


## Cell 7: The full smoke, as one
> The smoke test passed; the product was down. Two green probes above a service returning 500 on every real request, and a question from a corpus that no longer existed, its refusal counted as a pass. Since then: the golden question, the anonymous call refused, and one refusal per surface.


In [ ]:
# The smoke test passed. The product was down.
#
# This is real, it was found while writing this lesson, and it is the best
# argument in the course for what a smoke test is FOR.
PROBES = [
    ('GET  /health',   'returns {"status": "ok"}',        True),
    ('GET  /ready',    'warms the clients, returns ok',   True),
    ('POST /v1/query', 'AttributeError, HTTP 500',        False),
    ('POST /v1/stream', 'AttributeError, HTTP 500',       False),
]
print(f'  {"endpoint":16} {"what it did":34} up?')
for ep, what, up in PROBES:
    print(f'  {ep:16} {what:34} {"yes" if up else "NO"}')
print()
print('  Cloud Run watches /health. Kubernetes would watch /ready. Both were')
print('  green, the revision was serving, the dashboard was clean - and the only')
print('  two endpoints anybody actually calls returned 500.')
print()

# The cause, in one line.
BEFORE = "chunks = pack_chunks(chunks, settings.max_context_tokens, estimate_tokens)"
AFTER = "context, packed, dropped = pack_chunks(chunks, settings.max_context_tokens, ...)"
print(f'  was:  {BEFORE}')
print(f'  now:  {AFTER}')
print()
print('  pack_chunks returns THREE values - (context, packed, dropped) - and its')
print('  own docstring says so. The whole tuple was assigned to `chunks`, and the')
print('  next line iterated it and called .get() on the first element, a string.')
print()
print('  Nothing caught it. py_compile passes: it is valid Python. The import')
print('  check passes: the names all exist. Only RUNNING it fails, and nothing')
print('  ran it, because the smoke test asked the two questions that were fine.')
print()
print('  So the rule for the full smoke is not "add more checks". It is: check')
print('  the thing the product is FOR. DocuMind answers questions with citations.')
print('  A smoke test that never asks a question is testing a web server.')


In [ ]:
# THE FULL SMOKE, AS ONE. Seven scripts under deploy/smoke, one per surface, each configured by environment and each
# with at least one check that must REFUSE - the outsider on the MCP server, the chat, the media routes; nobody on the
# API. Run from the clone with the exports make smoke-all sets; PASS and FAIL lines collected, the refusal lines
# counted. A system that answers everybody passes a smoke made only of "does it work?".
env = {**os.environ, "DOCUMIND_PROJECT": PROJECT_ID, "DOCUMIND_API_URL": API_URL, "DOCUMIND_TENANT": TENANT,
       "DOCUMIND_IMPERSONATE_SA": MEMBER_SA, "DOCUMIND_OUTSIDER_SA": OUTSIDER_SA, "DOCUMIND_MCP_URL": MCP_URL,
       "DOCUMIND_CHAT_URL": CHAT_URL, "DOCUMIND_AGENT_URL": AGENT_URL, "DOCUMIND_GATEWAY_URL": GATEWAY_URL,
       "DOCUMIND_SLM_URL": SLM_URL, "SLM_REGION": REGION}
results = {}
for script in ("smoke", "smoke_mcp", "smoke_chat", "smoke_agent", "smoke_media", "smoke_gateway", "smoke_slm"):
    r = subprocess.run([sys.executable, f"{KIT}/deploy/smoke/{script}.py"], capture_output=True, text=True, env=env, cwd=f"{KIT}/deploy")
    lines = [l.strip() for l in (r.stdout + r.stderr).splitlines() if "[PASS]" in l or "[FAIL]" in l or "not set" in l]
    results[script] = (r.returncode, lines)
    print(f"== {script:14} exit {r.returncode}")
    for l in lines:
        print("   ", l[:110])
passes = sum(l.count("[PASS]") for _, ls in results.values() for l in ls)
fails = sum(l.count("[FAIL]") for _, ls in results.values() for l in ls)
refusals = [l for _, ls in results.values() for l in ls if "[PASS]" in l and any(w in l.lower() for w in ("refus", "403", "401", "outsider", "no token", "denied"))]
print(f"\n{passes} PASS, {fails} FAIL, {len(refusals)} of the passes are refusals")
red = [s for s, (rc, _) in results.items() if rc != 0]
print("red:", red or "none - the full smoke passes; gateway and slm go red until Module 11's deploy phase has run (make deploy-gateway, make deploy-slm SLM_STOCK=gemma3:4b)")
assert results["smoke"][0] == 0 and results["smoke_chat"][0] == 0 and results["smoke_mcp"][0] == 0 and refusals, "the sessions' surfaces must pass, refusals included"


### And before any of it: the pre-flight
Read-only, creates nothing, one line per thing `make up` will need - and every MISS line carries its fix.


In [ ]:
# THE PRE-FLIGHT, READ-ONLY. Before make up on a fresh project, smoke/preflight.sh asks for everything up will need and
# creates nothing: the four tools, terraform >= 1.9, a signed-in account, the project, billing, the state bucket, the
# twenty-one APIs, the Firestore client make roster needs. One line per check, ok or MISS, exit non-zero on any MISS -
# and every MISS line carries the command that fixes it. Run here against this project: Colab has gcloud and python
# but no terraform and no make, so two lines say MISS honestly and the exit code is 1. Then the checks, listed from
# the script itself, so the cell cannot describe a check the script does not make.
env = {**os.environ, "PROJECT": PROJECT_ID, "TFSTATE_BUCKET": f"{PROJECT_ID}-tfstate", "REGION": REGION}
r = subprocess.run(["sh", f"{KIT}/deploy/smoke/preflight.sh"], env=env, capture_output=True, text=True)
print(r.stdout.strip())
print(f"{chr(10)}exit {r.returncode} - the MISS lines above are this notebook's machine, not the project's, when they name a tool")
script = open(f"{KIT}/deploy/smoke/preflight.sh", encoding="utf-8").read()
checks = re.findall(r'ok +"([^"]+)"', script)
apis = re.search(r"for api in (.*?); do", script, re.S).group(1).split()
print(f"{chr(10)}{len(checks)} checks:", " | ".join(checks))
print(f"{len(apis)} APIs:", " ".join(apis))
assert "billing linked" in checks and "APIs enabled" in checks and "run" in apis and "firestore" in apis
print(f"{chr(10)}make preflight PROJECT=... TFSTATE_BUCKET=... is the same script from the Makefile; the runbook's first line")


## Cell 8: Turn it off, and know what comes back


In [ ]:
# TURN IT OFF - AND KNOW WHAT COMES BACK. make off floors the GPU services, the gateway and the UI to zero and deletes
# the cluster if one exists; the nightly job does the same at 23:00 whether anyone remembers. Read back here: every
# floor, the job's last execution, the scheduler. Then the honest inventory: what Terraform and the Makefile recreate
# from nothing (everything declared), and what they do NOT - the tuned endpoint (10.1's job, hours), the GGUF and the
# Modelfile (10.5), the demo-state revisions the replays flip to - which is why the lane is switched off, not destroyed,
# until the sessions are done, and why make down cannot finish on this project at all (the audit bucket's locked
# retention, Firestore's delete protection): a true zero is the project's deletion.
for name in ("documind-slm", "documind-vllm", "documind-gateway", "documind-ui", "documind-api", "documind-chat", "documind-mcp", "documind-agent", "documind-ingest"):
    s = service(name)
    print(f"  {name:18} {'absent' if not s else 'min-instances ' + s['min_instances']}")
print("  cluster           ", gcloud("container", "clusters", "list", "--format=value(name,status)") or "absent")
print("  nightly job       ", gcloud("scheduler", "jobs", "describe", "documind-off-nightly", "--location", REGION, "--format=value(state,schedule,lastAttemptTime)") or "not created yet")
print("  last execution    ", gcloud("run", "jobs", "executions", "list", "--job", "documind-off", "--region", REGION, "--limit", "1", "--format=value(metadata.name,status.completionTime)") or "none yet")
print()
print("recreated by make up   : every resource in the state (12.1), the roster, the corpus index (make ingest-corpus), the images (make build)")
print("recreated by nothing   : the tuned endpoint (10.1), the GGUF + Modelfile in the datasets bucket (10.5), the demo-state revisions the replays flip to (4.8, 7.2)")
print("make down stops at     : the audit bucket (retention locked five years), Firestore (delete protection) - a true zero is gcloud projects delete, after the sessions")


In [ ]:
# Turn it off. All of it. This is the last thing DocuMind teaches - updated for the lane that exists.
TEARDOWN = [
    ('Cloud Run min-instances > 0',  'make off  (or the 23:00 job: off.tf)',           '11.4: an idle warm L4 bills Rs 86,904 a month; a warm UI ~Rs 11,800'),
    ('GKE Autopilot cluster',        'make gke-down',                                  '11.5: deleting the WORKLOAD leaves the cluster fee; make off checks'),
    ('Vector Search index endpoint', 'undeploy, then delete (full profile only)',      'a DEPLOYED index bills whether or not you query it'),
    ('context caches',               'client.caches.delete(name=...)',                 '10.2: nothing expires a cache for you; storage bills by the hour'),
    ('a candidate revision',         'update-traffic --remove-tags candidate',         'a candidate answers, and bills, while it is tagged'),
    ('the lane at rest',             'nothing - it costs about Rs 200 a month',        'Cloud Run at zero, Firestore, Logging, the buckets'),
    ('the project itself',           'gcloud projects delete  (after the sessions)',   'the only step that guarantees Rs 0; make down cannot finish here'),
]
print(f'  {"what":30} {"how":46} why it is on this list')
for what, how, why in TEARDOWN:
    print(f'  {what:30} {how:46} {why}')
print()
print('  The expensive rows share a shape: the thing you deleted was not the thing that was billing.')
print('  A workload is not a cluster. A deployment is not an index endpoint. A cache is not a request.')
print('  The budget alert is the backstop, not the plan - it tells you AFTER the money is spent. The cap,')
print('  the job and the two left-warm alarms (12.1) are the plan. Run make off the same day as the demo,')
print('  not the Monday after: the gap between "it went well" and "somebody remembered" is the invoice.')


## Cell 9: Look how far


In [ ]:
# Twelve modules of BUILDING - and then a thirteenth that asks you to
# defend what you built. What DocuMind can do, and what each module added.
JOURNEY = [
    ( 1, 'a project, a client, one call',              'PROJECT_ID and a token count'),
    ( 2, 'prompts that survive a second reader',       'the system prompt'),
    ( 3, 'structure instead of vibes',                 'JSON that parses every time'),
    ( 4, 'retrieval that cites',                       'chunk_id, source_uri, page, quote'),
    ( 5, 'documents that are not text',                'Doc AI, and PII that gets caught'),
    ( 6, 'tools, and a loop that stops',               'one retrieve(), one contract'),
    ( 7, 'services that call each other safely',       'an ID token per call'),
    ( 8, 'an agent with a harness',                    'middleware, limits, a PII gate'),
    ( 9, 'more than text',                             'images, audio, video, a whiteboard'),
    (10, 'models you tuned and evaluated',             'a golden set with a number on it'),
    (11, 'a model you host yourself',                  'and the arithmetic to justify it'),
    (12, 'a product',                                  'shipped, guarded, observed, and off'),
]
for n, what, artefact in JOURNEY:
    print(f'  Module {n:2}  {what:44} {artefact}')
print()
print('  Read the right-hand column downward. Not one of those is a model')
print('  capability. They are all things you had to decide, build and check -')
print('  and they are what the difference between a demo and a product is made')
print('  of.')
print()
print('  DocuMind now takes a document a person uploaded two minutes ago,')
print('  answers a question about it with a citation you can click, refuses a')
print('  question it cannot ground, refuses a PERSON it cannot place, records')
print('  what it did without recording what was said, prices the answer in')
print('  rupees, ships through a pipeline nobody holds a key to, and turns off.')
print()
print('  Module 13 asks you to defend it. Not to build more.')


## Where this goes
- **4.8**'s gate and **12.7**'s release are what the smokes stand in front of; **12.1** is the empty plan the switched-off lane still satisfies.
- The sessions: the runbook's pre-flight is `make smoke-all`, and its last line is `make off`.

## ✅ Lesson 12.8 complete
- ✅ One verifier, four answers: 200, 403 from the roster, refused by the platform, 401 on a forged assertion
- ✅ The chat surface: answered as a member; a tenant in the body changed nothing, and the row named the caller's
- ✅ The roster read both ways for three tenants, the member and the outsider
- ✅ Seven smokes run as one; the refusal lines counted
- ✅ The pre-flight run read-only against the project; its checks and its API list read from the script
- ✅ The lane turned off: every floor read back; what Terraform recreates and what nothing does


## The files this lesson owns
Below are the six heredocs the extractor places: the one verifier, the roster read both ways, the chat agent (the surface that had no identity), its three brains behind one switch, Cloud SQL for the conversation that survives a restart (full profile), and the chat deploy with IAP enabled after the service exists. Unchanged by the rebuild; the story above asked the verifier four ways and ran the smokes against the surfaces these files build.


In [ ]:
IAP_PY = r'''
"""One IAP verifier, for every surface. Lesson 12.8.

WHY THIS FILE EXISTS. DocuMind had FOUR surfaces and four different answers to "who is asking":

    documind-api    rag-api/auth.py     google.oauth2.id_token.verify_token, certs_url=
                                        .../iap/verify/public_key, fails closed on a missing
                                        audience, then a Firestore roster check.
    documind-admin  admin/auth.py       the same library and endpoint, fails closed the same
                                        way, then an ADMIN_EMAILS allowlist.
    documind-ui     frontend/auth.py    a DIFFERENT library - PyJWT's PyJWKClient against
                                        .../iap/verify/public_key-jwk, algorithms=["ES256"],
                                        explicit issuer - and it does NOT fail closed when
                                        IAP_AUDIENCE is unset.
    documind-chat   chat/agent.py       nothing at all. tenant_id and user_id arrive in the
                                        request body, so the caller picks the tenant.

None of the first three is wrong on its own. The problem is that there are three: a fix now has
to be made three times, and it will be made once. That is the same argument deploy/UNOWNED.md
already makes about shared/pii.py - "two lists that drift is the worst kind of compliance bug:
the scan misses a type, the dashboard reports zero findings, and both look correct". Swap "list"
for "verifier".

WHAT VERIFICATION HAS TO ESTABLISH, and all four must hold:

    signature   signed by Google IAP, against the IAP key set - not the general OAuth certs.
    audience    minted for THIS service. Skip it and a token for ANY IAP-protected service in
                the project is accepted. This is the check people leave out.
    issuer      https://cloud.google.com/iap
    expiry      still valid. The library does this for you if you let it.

Reading `x-goog-authenticated-user-email` instead is not a shortcut, it is the absence of a
check: anything that can reach the service can set a header.

AUDIENCE, PRECISELY. For a Cloud Run service with IAP enabled directly, the audience is

    /projects/PROJECT_NUMBER/locations/REGION/services/SERVICE_NAME

Note the leading slash and the PROJECT NUMBER, not the project id. Getting this wrong produces
a permanent 401 that looks exactly like a broken login.

ACCEPT_AUDIENCES IS A LIST, ON PURPOSE. documind-api is not behind IAP - nothing reaches it
from a browser - so the assertion it sees was minted for whichever SURFACE forwarded it, and
there is more than one (documind-ui and documind-chat). A service that accepts forwarded
assertions must name every audience it will accept, and must accept no others.
"""
from __future__ import annotations

import os
from functools import lru_cache

# google-auth is imported INSIDE verify(), not here. The allowlist and audience logic below
# is where the bugs were, and it should be testable without a cloud SDK on the machine -
# a security helper nobody can unit-test is a security helper nobody unit-tests.

# IAP signs with its OWN key set. The general Google OAuth certs will not verify these.
IAP_CERTS = "https://www.gstatic.com/iap/verify/public_key"
IAP_ISSUER = "https://cloud.google.com/iap"
ASSERTION_HEADER = "x-goog-iap-jwt-assertion"


class IapError(Exception):
    """Verification failed. The caller decides whether that is a 401 or a 403."""


@lru_cache(maxsize=1)
def accepted_audiences() -> tuple[str, ...]:
    """Every audience this service will accept, from IAP_AUDIENCE (comma-separated).

    Empty is FAIL CLOSED, not "accept anything". frontend/auth.py used to pass the unset
    value straight to the library and rely on whatever it did with audience=None - a
    security property nobody had decided, resolved by a dependency's default.
    """
    raw = os.environ.get("IAP_AUDIENCE", "")
    return tuple(a.strip() for a in raw.split(",") if a.strip())


def verify(assertion: str | None) -> dict:
    """Return the verified claims, or raise IapError. Never returns unverified data."""
    auds = accepted_audiences()
    if not auds:
        raise IapError("IAP_AUDIENCE is not configured; refusing to authenticate")
    if not assertion:
        raise IapError(f"no {ASSERTION_HEADER} - this request did not come through IAP")

    from google.auth.transport import requests as g_requests
    from google.oauth2 import id_token

    last = None
    for aud in auds:
        try:
            claims = id_token.verify_token(
                assertion, g_requests.Request(), audience=aud, certs_url=IAP_CERTS)
        except Exception as e:            # noqa: BLE001 - any failure is a refusal
            last = e
            continue
        if claims.get("iss") != IAP_ISSUER:
            raise IapError(f"issuer is {claims.get('iss')!r}, not IAP")
        email = (claims.get("email") or "").lower()
        if not email:
            raise IapError("the assertion carries no email")
        return {"email": email, "sub": claims.get("sub"), "aud": aud}
    raise IapError(f"invalid assertion: {type(last).__name__ if last else 'no audience matched'}")


def email_from(headers) -> str:
    """The verified caller's email, from a mapping of request headers.

    Accepts anything with .get() - FastAPI's request.headers and Streamlit's
    st.context.headers both qualify, which is why this signature and not a framework type.
    """
    return verify(headers.get(ASSERTION_HEADER))["email"]


def parse_allowlist(raw: str | None) -> frozenset[str]:
    """Split a comma-separated allowlist, DROPPING empties.

    This exists because of a real bug. frontend/auth.py did

        ADMIN_EMAILS = set(os.getenv("ADMIN_EMAILS", "").split(","))

    and ''.split(',') is [''], not []. So an unset allowlist contained the empty string, and
    is_admin() returned True for any user whose email was missing - an unset allowlist meaning
    "anybody without an email" instead of "nobody". admin/auth.py parsed the same variable with
    `if e.strip()` and was correct. One function, so there is one behaviour.
    """
    return frozenset(e.strip().lower() for e in (raw or "").split(",") if e.strip())


def is_allowed(email: str, emails: frozenset[str], domains: frozenset[str] = frozenset()) -> bool:
    """Membership of an allowlist. An EMPTY allowlist allows nobody."""
    email = (email or "").lower()
    if not email or "@" not in email:
        return False                      # no identity is not a match, it is a refusal
    if not emails and not domains:
        return False
    return email in emails or email.rsplit("@", 1)[-1] in domains


# ---------------------------------------------------------------------------- the other leg
# Gap G4 (2026-09-05). Not every caller has a person behind it. An agent in a notebook,
# run_eval.py, smoke.py: nobody signed in, so IAP minted nothing. What they DO have is the
# Google ID token they minted for this service's URL (lesson 7.3) - the same token Cloud Run's
# IAM ingress checked roles/run.invoker on. Re-verifying it here is what lets the email inside
# be trusted as the caller. It establishes WHO, not WHETHER: the account still has to be on the
# tenant's roster (shared/tenancy.py).


def bearer_email(headers, audience: str) -> str:
    """The identity behind `Authorization: Bearer <Google ID token>`, verified for `audience`."""
    auth = headers.get("authorization") or headers.get("Authorization") or ""
    if not auth.lower().startswith("bearer "):
        raise IapError("no Authorization: Bearer token")
    if not audience:
        raise IapError("SELF_URL is not configured; refusing to verify bearer tokens")

    from google.auth.transport import requests as g_requests
    from google.oauth2 import id_token

    try:
        claims = id_token.verify_oauth2_token(auth[7:].strip(), g_requests.Request(),
                                              audience=audience)
    except Exception as e:                # noqa: BLE001 - any failure is a refusal
        raise IapError(f"invalid bearer token: {type(e).__name__}")
    email = (claims.get("email") or "").lower()
    if not email or not claims.get("email_verified", True):
        raise IapError("the bearer token carries no verified email")
    return email


def identity(headers, *, bearer_audience: str | None = None) -> dict:
    """Who is this request for? The assertion first, the bearer token second, never a header.

    A surface (documind-ui, documind-chat) forwards the person's IAP assertion beside its own
    ID token. The assertion wins because it names the PERSON; the token only proves the surface
    was allowed to call. With no assertion and a bearer audience configured, the token's own
    identity is the caller. With neither, refuse - there is nobody to be.
    """
    assertion = headers.get(ASSERTION_HEADER)
    if assertion:
        claims = verify(assertion)
        return {"email": claims["email"], "via": "iap", "aud": claims["aud"]}
    if bearer_audience:
        return {"email": bearer_email(headers, bearer_audience), "via": "iam",
                "aud": bearer_audience}
    raise IapError(f"no {ASSERTION_HEADER} and no bearer audience configured - refusing")
'''

with open('iap.py', 'w') as f: f.write(IAP_PY)
print('iap.py:', len(IAP_PY.splitlines()), 'lines')


In [ ]:
TENANCY_PY = r'''
"""One tenant roster, read one way. Lesson 12.8.

The roster is Firestore, at `tenants/{tenant_id}/members/{email}`. Three surfaces need it and
each had arrived at its own version:

    rag-api/auth.py       is_member(email, tenant) - a POINT lookup. It is given the tenant and
                          asks "is this person in it?". Right for a service that receives a
                          tenant_id on the request and must authorise it.
    frontend/auth.py      tenant_for(email) - a REVERSE lookup by collection group. It is given
                          a person and asks "which tenant?". Right for a browser surface where
                          nobody sends a tenant at all.
    chat/agent.py         neither. It believed the request body.

Both directions are legitimate and both are here, so the next surface picks one instead of
writing a third. The member document is keyed by EMAIL so the point lookup is a single read,
and carries `email` as a FIELD so the reverse lookup is one collection-group query rather than
a scan of every tenant - that shape is load-bearing for both functions and is why neither is
implemented in terms of the other.

WHY A PERSON HAS ONE TENANT HERE. The reverse lookup returns the first match. DocuMind's model
is that a person belongs to one customer; a consultant working for two would need this to
return a list and every caller to choose, which is a product decision, not a code change. Said
out loud because "it returns the first row" is otherwise indistinguishable from a bug.
"""
from __future__ import annotations

from functools import lru_cache


@lru_cache(maxsize=1)
def _db():
    from google.cloud import firestore
    return firestore.Client()


def is_member(email: str, tenant_id: str) -> bool:
    """Is this person on THIS tenant's roster? One document read."""
    if not email or not tenant_id:
        return False
    doc = (_db().collection("tenants").document(tenant_id)
           .collection("members").document(email.lower()).get())
    return doc.exists


def tenant_for(email: str) -> str | None:
    """Which tenant does this person belong to? None if nobody's.

    None is the honest answer for a verified user who is on no roster, and the caller
    should turn it into a 403 - they are authenticated and not authorised. Returning a
    default tenant here would be the same class of bug as trusting a header.
    """
    if not email:
        return None
    hits = (_db().collection_group("members")
            .where("email", "==", email.lower()).limit(1).get())
    for doc in hits:
        return doc.reference.parent.parent.id      # tenants/{THIS}/members/{email}
    return None


# ---- the write side, for the operator ----------------------------------------------------
# Nothing above WRITES the roster; a service that could would be a service that could enrol
# itself. Membership is an operator's act - `make roster` in deploy/ runs this - and the
# document shape is exactly what the two readers above depend on: keyed by email, with
# `email` as a field.

def add_member(tenant_id: str, email: str) -> None:
    """Put a person on a tenant's roster. Idempotent: the same document, the same fields."""
    from google.cloud import firestore
    (_db().collection("tenants").document(tenant_id)
         .collection("members").document(email.lower())
         .set({"email": email.lower(), "added_at": firestore.SERVER_TIMESTAMP}))


def list_members(tenant_id: str) -> list[str]:
    return sorted(d.id for d in _db().collection("tenants").document(tenant_id)
                  .collection("members").stream())


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser(description="the tenant roster, from the operator's side")
    sub = ap.add_subparsers(dest="cmd", required=True)
    a = sub.add_parser("add", help="put a person (or a service account) on a tenant's roster")
    a.add_argument("tenant"); a.add_argument("email")
    l = sub.add_parser("list", help="who is on a tenant's roster")
    l.add_argument("tenant")
    args = ap.parse_args()
    if args.cmd == "add":
        add_member(args.tenant, args.email)
        print(f"{args.email.lower()} is on {args.tenant}")
    else:
        for m in list_members(args.tenant):
            print(m)
'''

with open('tenancy.py', 'w') as f: f.write(TENANCY_PY)
print('tenancy.py:', len(TENANCY_PY.splitlines()), 'lines')


In [ ]:
CHAT_AGENT_PY = r'''
"""DocuMind chat service - the surface, the identity, the memory, and the brain switch.

This is the production half of lesson 6.4, grown by 8.5 (a checkpointer), 8.7 (three brains
over one tool) and 12.8 (one identity). The brains live in `brains.py`; this file decides who
is asking, which tenant they are, which conversation this is, and which brain answers - then
logs one row saying so.

Six things here are load-bearing and easy to get wrong:

0. THE TENANT IS NOT A REQUEST FIELD. It used to be, and that made every document in every
   tenant readable with one curl. It is now looked up from the Firestore roster using the
   email on the verified IAP assertion. The retriever's tenant filter was always correct;
   it was filtering on a value the caller chose.

1. `wrap_tool_call` must return a `ToolMessage` or a `Command`, never a bare dict (brains.py).
2. `tenant_id` reaches the tools through the `ToolRuntime` the framework injects - its
   `.context` is the dict passed to `invoke(..., context=...)` - so it never appears in the
   schema the model reads. If it were an ordinary parameter the model would choose the tenant,
   which is a cross-tenant read wearing the costume of a tool argument. It was first written
   as `Annotated[str, InjectedToolArg]`, which hides an argument from the model and does
   nothing else: ToolNode never fills it, the tool ran with tenant_id="" and rag-api answered
   403 (proven offline 2026-09-05, gap G4). The person's IAP assertion travels the same way:
   read off this request, put in the context, forwarded to rag-api by the one `retrieve()`
   beside the service's own ID token - never sent as a header anyone could set.
3. THE PROFILE SWITCH IS 6.4's, UNCHANGED (gap G3). `shared/profile.py` decides the model
   (Gemini on Vertex AI, or Ollama gemma3:4b) and `documind_tools.retrieve()` decides the store
   (rag-api, or a Chroma directory). With DOCUMIND_PROFILE=local there is no IAP in front of
   this service either, so identity comes from LOCAL_USER / LOCAL_TENANT - a dev-only bypass
   that is explicit, named, and impossible to reach with the profile set to gcp.
4. A CONVERSATION SURVIVES A RESTART (gap G5, lesson 8.5). Every brain gets the checkpointer
   the profile chooses: SqliteSaver on the laptop, PostgresSaver on Cloud SQL in production,
   InMemorySaver only when CHECKPOINT_DSN=memory says so out loud. The thread id is
   `tenant:user:session` - 8.5's boundary - built by the SERVER from the verified identity, so
   a session id from the request can name a conversation but never a tenant. The connection
   is opened once, in the lifespan, and outlives every request: 8.5's trap is returning from
   inside `from_conn_string()`. `setup()` is NOT called here - `migrate.py` is a one-off job.
5. THE BRAIN IS A SWITCH, NOT A FORK (gap G6, lesson 8.7). DOCUMIND_BRAIN picks the default;
   `brain` on the request (allow-listed) overrides it per turn - that is the UI's radio. Brains
   are built lazily and cached for the process; one that is not installed is a 501, not a
   crash at startup. The usage row names the brain, and rag-api's row does too.

Verified 2026-09-04 against langchain 1.4.0 / langchain-google-genai 4.4.0. The bearer leg (SELF_URL)
arrived with Module 8 on the lane, 2026-09-08: it is what lets the chat service run on the lean profile
and be smoke-tested from a shell, and it changes nothing for a person behind IAP.
"""
from __future__ import annotations

import json
import logging
import os
import time
from contextlib import ExitStack, asynccontextmanager
from typing import Literal, Optional

from fastapi import Depends, FastAPI, HTTPException, Request
from pydantic import BaseModel, Field

from brains import BRAINS, DEFAULT_BRAIN, build as build_brain
from shared import iap
from shared.profile import PROFILE
from shared.tenancy import tenant_for

logger = logging.getLogger("documind.chat.agent")

CHECKPOINT_DSN = os.environ.get("CHECKPOINT_DSN", "")
# This service's own URL - the audience an agent, a smoke test or 8.7's notebook mints an ID
# token for when there is no person and no IAP assertion (shared/iap.identity, the bearer leg).
SELF_URL = os.environ.get("SELF_URL", "").rstrip("/")


def build_checkpointer(stack: ExitStack):
    """8.5's three lanes. `stack` keeps the connection open for the life of the process."""
    if PROFILE == "local":
        # A file on disk. Survives a restart of the process - and is per-instance, which is
        # why it is the laptop lane and not a small production one.
        from langgraph.checkpoint.sqlite import SqliteSaver
        path = os.environ.get("DOCUMIND_THREADS_DB", "./documind_threads.db")
        return stack.enter_context(SqliteSaver.from_conn_string(path))
    if CHECKPOINT_DSN == "memory":
        from langgraph.checkpoint.memory import InMemorySaver
        logger.warning("CHECKPOINT_DSN=memory: conversations die with the instance (8.5). "
                       "Tests only - never a deployment.")
        return InMemorySaver()
    if not CHECKPOINT_DSN:
        raise RuntimeError("CHECKPOINT_DSN is not set. terraform/cloudsql.tf creates the instance "
                           "and the secret; --set-secrets mounts it (commands/lesson-12.8.sh).")
    from langgraph.checkpoint.postgres import PostgresSaver
    from psycopg.rows import dict_row
    from psycopg_pool import ConnectionPool

    # Cloud Run reaches Cloud SQL over a unix socket; the DSN carries host=/cloudsql/... (8.5).
    # autocommit, prepare_threshold=0 and dict_row are what the checkpointer requires of psycopg.
    pool = stack.enter_context(ConnectionPool(
        CHECKPOINT_DSN, min_size=1, max_size=4, open=True,
        kwargs={"autocommit": True, "prepare_threshold": 0, "row_factory": dict_row}))
    return PostgresSaver(pool)      # setup() deliberately absent - see migrate.py


def thread_config(tenant_id: str, user_id: str, session_id: str) -> dict:
    """The thread id IS the tenancy boundary (8.5): tenant first, so nothing after it can forge
    another tenant; ':' banned in every part, so two users cannot collide on one thread."""
    parts = (tenant_id, user_id, session_id)
    if not all(parts):
        raise HTTPException(500, "tenant, user and session ids must all be non-empty")
    if any(":" in p for p in parts):
        raise HTTPException(400, "ids must not contain ':'")
    return {"configurable": {"thread_id": f"{tenant_id}:{user_id}:{session_id}"}}


def brain_for(app: FastAPI, name: str):
    """Build once, keep for the process. A framework that is not installed is a 501."""
    if name not in app.state.brains:
        try:
            app.state.brains[name] = build_brain(name, app.state.checkpointer)
        except ImportError as exc:
            raise HTTPException(501, f"brain {name!r} is not installed in this image: {exc.name}")
    return app.state.brains[name]


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Open the checkpointer once; brains are built on first use and share it."""
    with ExitStack() as stack:
        app.state.checkpointer = build_checkpointer(stack)
        app.state.brains = {}
        yield


app = FastAPI(title="documind-chat", lifespan=lifespan)


class ChatRequest(BaseModel):
    """The question, which conversation it belongs to, and optionally which brain - nothing else.

    tenant_id and user_id used to be fields here, and the agent believed them. Removing them
    is stronger than validating them: a validation is something a later edit can loosen, and
    a field that does not exist is not. There is now nowhere in this request to lie. session_id
    names a conversation INSIDE the caller's own tenant and user - the server prefixes both."""

    question: str = Field(min_length=1, max_length=4000)
    session_id: str = Field(default="default", pattern=r"^[A-Za-z0-9_-]{1,64}$")
    brain: Optional[Literal["langchain", "langgraph", "adk", "direct"]] = None


def caller(request: Request) -> dict:
    """Who is asking, verified - one implementation, shared/iap.py, four surfaces.

    Returns the email AND the raw assertion: the assertion is forwarded to rag-api so the
    retrieval is made in the person's name, not the service's."""
    if PROFILE == "local":
        # No IAP on a laptop. Named, explicit, and unreachable once the profile is gcp.
        return {"email": os.environ.get("LOCAL_USER", "dev@documind.local"), "assertion": None}
    try:
        # The assertion first - a person, through the UI, and it is forwarded to rag-api so the
        # retrieval is made in their name. The bearer token second - an agent, make smoke-chat,
        # 8.7's notebook: minted for THIS service's URL, verified here, and the roster still decides
        # (7.1 taught the leg; 12.8 shipped it on every surface). One implementation: shared/iap.py.
        who = iap.identity(request.headers, bearer_audience=SELF_URL or None)
        return {"email": who["email"],
                "assertion": request.headers.get(iap.ASSERTION_HEADER) if who["via"] == "iap" else None}
    except iap.IapError as e:
        # 401, not 403: we do not know who this is. 403 is for somebody we DO know and
        # will not serve, and the difference matters when you are reading logs at 3am.
        raise HTTPException(401, str(e))


@app.get("/health")
def health() -> dict:
    return {"status": "ok", "profile": PROFILE, "brains": list(BRAINS), "default_brain": DEFAULT_BRAIN}


@app.post("/v1/chat")
def chat(req: ChatRequest, request: Request, user=Depends(caller)) -> dict:
    # The tenant is LOOKED UP, never received. Same Firestore roster rag-api checks in
    # enforce_membership() and the frontend reads in tenant_for() - one roster, three
    # surfaces, so a membership change takes effect everywhere at once. The local lane has
    # no roster and no Firestore: LOCAL_TENANT names the one tenant the Chroma directory holds.
    if PROFILE == "local":
        tenant_id = os.environ.get("LOCAL_TENANT", "acme")
    else:
        tenant_id = tenant_for(user["email"])
    if tenant_id is None:
        raise HTTPException(403, "not a member of any tenant")

    name = req.brain or DEFAULT_BRAIN
    brain = brain_for(request.app, name)

    # The tenant, the assertion and the brain travel in the runtime context, NOT in the
    # question and NOT in the tool schema. tools.py reads them from the ToolRuntime the
    # framework injects. The thread id carries the checkpoint (8.5): same tenant, same user,
    # same session -> the conversation continues, on whichever instance answers.
    t0 = time.monotonic()
    out = brain.answer(
        req.question,
        config=thread_config(tenant_id, user["email"], req.session_id),
        context={"tenant_id": tenant_id, "user_id": user["email"],
                 "assertion": user["assertion"] or "", "brain": name},
    )
    latency_ms = int((time.monotonic() - t0) * 1000)
    # One row per turn, in the shape 12.3's sink collects: WHICH brain answered is the field
    # 8.7's cost comparison needs and the one the plan's M12 gate asks for.
    logger.info(json.dumps({"event": "chat", "surface": "chat", "brain": name,
                            "tenant": tenant_id, "user": user["email"],
                            "session_id": req.session_id, "latency_ms": latency_ms,
                            "tool_calls": out.get("tool_calls", []),
                            "refusals": out.get("refusals", [])}))
    return {**out, "brain": name, "session_id": req.session_id, "latency_ms": latency_ms}
'''

with open('agent.py', 'w') as f: f.write(CHAT_AGENT_PY)
print('agent.py:', len(CHAT_AGENT_PY.splitlines()), 'lines')


In [ ]:
CHAT_BRAINS_PY = r'''
"""DocuMind chat service - three brains over one tool. Lessons 8.1-8.7, gap G6.

8.7's harness put one question through three agent runtimes and printed three cost lines, and
the finding was that every difference was the HARNESS - the tool was identical. This module is
that harness, shipped: the same `tools.py` (which adapts the one `documind_tools.retrieve()`),
the same checkpointer, the same thread id, four ways to drive a model at them:

    langchain   6.4's create_agent + the guard as middleware          (the default)
    langgraph   8.5/8.6's hand-built StateGraph - agent, tools, refuse (more control, nothing done for you)
    adk         8.1-8.4's LlmAgent + Runner, the guard as before_tool_callback
    direct      no loop at all: one retrieve(), rag-api's own grounded answer - the floor to compare against

Which one answers is DOCUMIND_BRAIN (the deploy) or the `brain` field on the request (the UI's
radio, 12.4), allow-listed to these four names. `agent.py` dispatches and logs the brain on
every usage row; `documind_tools.retrieve(brain=)` carries it to rag-api so ITS row has it too.

Two honest limits. The ADK brain keeps its session in ADK's own session service, not the
LangGraph checkpointer - 8.3's Agent Engine sessions are its production answer; here it uses
DatabaseSessionService on the same Cloud SQL when CHECKPOINT_DSN is set, InMemorySessionService
otherwise. And each brain imports its framework lazily, so a deployment that does not install
google-adk simply reports that brain as unavailable (501), rather than failing to start.

Verified offline 2026-09-05: the LangGraph brain's loop, refuse path, context delivery and
checkpoint (tools/check_auth_wiring.py). The ADK brain follows 8.7 cell 12 (google-adk 2.8.0).
"""
from __future__ import annotations

import asyncio
import contextvars
import json
import logging
import os

from langchain_core.messages import SystemMessage, ToolMessage

from shared import documind_tools
from shared.profile import LOCAL_MODEL, PROFILE, build_llm
from tools import BLOCKED, TIMEOUTS, TOOLS

logger = logging.getLogger("documind.chat.brains")

BRAINS = ("langchain", "langgraph", "adk", "direct")
DEFAULT_BRAIN = os.environ.get("DOCUMIND_BRAIN", "langchain")
MODEL = os.environ.get("CHAT_MODEL", "gemini-3.6-flash")

SYSTEM = (
    "You are DocuMind AI, a document intelligence assistant. "
    "Use the tools for document questions. When estimating costs, search first so the page "
    "count is real rather than guessed. If a tool refuses, say so plainly and explain what "
    "approval is needed - never claim the action was taken."
)


def _summary(messages) -> dict:
    """The same three keys from every brain, so agent.py and the UI never branch on the brain."""
    return {
        "answer": messages[-1].content if messages else "",
        "tool_calls": [tc["name"] for m in messages for tc in (getattr(m, "tool_calls", None) or [])],
        "refusals": [m.name for m in messages if isinstance(m, ToolMessage) and m.status == "error"],
    }


# ----------------------------------------------------------------------------- 1. langchain
def _guard_middleware():
    """6.4's guard, as create_agent middleware. Built here so langchain is imported lazily."""
    import time

    from langchain.agents.middleware import AgentMiddleware

    class GuardMiddleware(AgentMiddleware):
        """Refuse blocked operations, time every call, record all of them."""

        def wrap_tool_call(self, request, handler):
            name = request.tool_call["name"]
            if name in BLOCKED:
                logger.warning("refused %s (blocked list)", name)
                return ToolMessage(content=json.dumps({"error": f"{name} requires manual approval"}),
                                   name=name, tool_call_id=request.tool_call["id"], status="error")
            started = time.monotonic()
            try:
                return handler(request)
            finally:
                elapsed = time.monotonic() - started
                budget = TIMEOUTS.get(name, 30)
                (logger.warning if elapsed > budget else logger.info)(
                    "%s took %.2fs (budget %ss)", name, elapsed, budget)

    return GuardMiddleware()


class LangChainBrain:
    name = "langchain"

    def __init__(self, checkpointer, llm=None):
        from langchain.agents import create_agent

        self.agent = create_agent(model=llm or build_llm(), tools=TOOLS, system_prompt=SYSTEM,
                                  middleware=[_guard_middleware()], checkpointer=checkpointer)

    def answer(self, question: str, *, config: dict, context: dict) -> dict:
        out = self.agent.invoke({"messages": [{"role": "user", "content": question}]},
                                config=config, context=context)
        return _summary(out["messages"])


# ----------------------------------------------------------------------------- 2. langgraph
class LangGraphBrain:
    """8.7 cell 20's graph, plus the refuse path 8.6 drew: the harness is edges and nodes."""

    name = "langgraph"

    def __init__(self, checkpointer, llm=None):
        from langgraph.graph import END, START, MessagesState, StateGraph
        from langgraph.prebuilt import ToolNode

        model = (llm or build_llm()).bind_tools(TOOLS)

        def agent(state):
            # The system prompt is prepended per call, never stored - the checkpoint holds the
            # conversation, not the instructions, so a prompt change applies to old threads too.
            return {"messages": [model.invoke([SystemMessage(content=SYSTEM)] + state["messages"])]}

        def route(state):
            calls = getattr(state["messages"][-1], "tool_calls", None) or []
            if not calls:
                return END
            return "refuse" if any(c["name"] in BLOCKED for c in calls) else "tools"

        def refuse(state):
            # A turn that asks for a blocked tool is refused whole: every call in it gets an
            # error ToolMessage, the model reads them and explains. Same seam as GuardMiddleware.
            last = state["messages"][-1]
            return {"messages": [
                ToolMessage(content=json.dumps({"error": f"{c['name']} requires manual approval"}),
                            name=c["name"], tool_call_id=c["id"], status="error")
                for c in last.tool_calls]}

        g = StateGraph(MessagesState, context_schema=dict)
        g.add_node("agent", agent)
        g.add_node("tools", ToolNode(TOOLS))
        g.add_node("refuse", refuse)
        g.add_edge(START, "agent")
        g.add_conditional_edges("agent", route, {"tools": "tools", "refuse": "refuse", END: END})
        g.add_edge("tools", "agent")
        g.add_edge("refuse", "agent")
        self.graph = g.compile(checkpointer=checkpointer)

    def answer(self, question: str, *, config: dict, context: dict) -> dict:
        out = self.graph.invoke({"messages": [{"role": "user", "content": question}]},
                                config=config, context=context)
        return _summary(out["messages"])


# ----------------------------------------------------------------------------- 3. adk
# The request's tenant and assertion, for ADK's before_tool_callback. A ContextVar, not an
# attribute: the sync endpoint runs in a thread pool, and asyncio.run() copies the context into
# the task it creates, so two concurrent requests cannot see each other's tenant.
_adk_ctx: contextvars.ContextVar[dict] = contextvars.ContextVar("documind_adk_ctx", default={})


class AdkBrain:
    name = "adk"

    def __init__(self, checkpointer=None, llm=None):
        from google.adk.agents import LlmAgent
        from google.adk.agents.run_config import RunConfig
        from google.adk.apps import App
        from google.adk.runners import Runner
        from google.adk.tools import FunctionTool

        def guard_tool(tool, args, tool_context):
            """8.7 cell 12: return a dict to REPLACE the call, None to let it through."""
            if tool.name in BLOCKED:
                return {"error": f"{tool.name} is not reachable from a model turn"}
            ctx = _adk_ctx.get()
            args["tenant_id"] = ctx.get("tenant_id", "")        # never the model's choice
            if ctx.get("assertion"):
                args["assertion"] = ctx["assertion"]
            args["brain"] = "adk"
            return None

        if PROFILE == "local":
            from google.adk.models.lite_llm import LiteLlm
            model = LiteLlm(model=f"ollama_chat/{LOCAL_MODEL}")
        else:
            # ADK's Gemini client reads these; global because 3.x generation is not regional.
            os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "1")
            os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "global")
            model = MODEL

        root = LlmAgent(name="documind_adk", model=model, instruction=SYSTEM,
                        tools=[FunctionTool(documind_tools.retrieve),
                               FunctionTool(documind_tools.calculate_processing_cost)],
                        before_tool_callback=guard_tool)
        self.svc = self._sessions()
        self.runner = Runner(app=App(name="documind", root_agent=root), session_service=self.svc)
        self.run_config = RunConfig(max_llm_calls=int(os.environ.get("ADK_MAX_LLM_CALLS", "12")))

    @staticmethod
    def _sessions():
        """ADK keeps its own sessions. On the same Cloud SQL when there is one, in memory otherwise."""
        dsn = os.environ.get("CHECKPOINT_DSN", "")
        if dsn.startswith("postgresql://"):
            try:
                from google.adk.sessions import DatabaseSessionService
                return DatabaseSessionService(db_url=dsn.replace("postgresql://", "postgresql+psycopg://", 1))
            except Exception as exc:                          # noqa: BLE001
                logger.warning("ADK DatabaseSessionService unavailable (%s); using memory", exc)
        from google.adk.sessions import InMemorySessionService
        return InMemorySessionService()

    def answer(self, question: str, *, config: dict, context: dict) -> dict:
        session_id = config["configurable"]["thread_id"]     # tenant:user:session - the same boundary
        user_id = context.get("user_id") or "user"
        token = _adk_ctx.set(context)
        try:
            return asyncio.run(self._run(question, user_id, session_id))
        finally:
            _adk_ctx.reset(token)

    async def _run(self, question: str, user_id: str, session_id: str) -> dict:
        from google.genai import types as gt

        sess = await self.svc.get_session(app_name="documind", user_id=user_id, session_id=session_id)
        if sess is None:
            sess = await self.svc.create_session(app_name="documind", user_id=user_id, session_id=session_id)
        answer, tool_calls, refusals = "", [], []
        async for ev in self.runner.run_async(
                user_id=user_id, session_id=sess.id,
                new_message=gt.Content(role="user", parts=[gt.Part(text=question)]),
                run_config=self.run_config):
            for call in (ev.get_function_calls() or []):
                tool_calls.append(call.name)
                if call.name in BLOCKED:
                    refusals.append(call.name)
            if ev.is_final_response() and ev.content and ev.content.parts:
                answer = "".join(p.text or "" for p in ev.content.parts)
        return {"answer": answer, "tool_calls": tool_calls, "refusals": refusals}


# ----------------------------------------------------------------------------- 4. direct
class DirectBrain:
    """No loop. One retrieve(), and rag-api's own grounded answer comes back with it - the
    cheapest brain, and the one the other three have to beat to justify their harness."""

    name = "direct"

    def __init__(self, checkpointer=None, llm=None):
        self.llm = llm

    def answer(self, question: str, *, config: dict, context: dict) -> dict:
        r = documind_tools.retrieve(question, tenant_id=context.get("tenant_id", ""), top_k=5,
                                    assertion=context.get("assertion") or None, brain="direct")
        if "error" in r:
            return {"answer": r["error"], "tool_calls": ["retrieve"], "refusals": [], "citations": []}
        if r.get("answer"):
            return {"answer": r["answer"], "tool_calls": ["retrieve"], "refusals": [],
                    "citations": r["citations"]}
        # The local lane has no generator behind retrieve(): one grounded call to the profile's model.
        ctx = "\n".join(f"[Source {i}] {c['quote']}" for i, c in enumerate(r["citations"], 1))
        msg = (self.llm or build_llm()).invoke(
            f"{SYSTEM}\nAnswer only from the context and cite [Source N].\n\nContext:\n{ctx}\n\nQuestion: {question}")
        return {"answer": getattr(msg, "content", str(msg)), "tool_calls": ["retrieve"],
                "refusals": [], "citations": r["citations"]}


_REGISTRY = {"langchain": LangChainBrain, "langgraph": LangGraphBrain, "adk": AdkBrain, "direct": DirectBrain}


def build(name: str, checkpointer, llm=None):
    """One brain, by name. ImportError means that framework is not installed in this image."""
    if name not in BRAINS:
        raise ValueError(f"unknown brain {name!r}; one of {BRAINS}")
    return _REGISTRY[name](checkpointer, llm)
'''

with open('brains.py', 'w') as f: f.write(CHAT_BRAINS_PY)
print('brains.py:', len(CHAT_BRAINS_PY.splitlines()), 'lines')


In [ ]:
CLOUDSQL_TF = r'''
# Cloud SQL for the chat checkpointer. Lessons 8.5 and 12.8 - gap G5.
#
# WHY. A conversation that survives a restart is Module 8's gate, and 8.5 was precise about the
# three lanes: InMemorySaver dies with the process, SqliteSaver is a file on ONE instance, and
# only a database outlives a deploy, a scale-to-zero and an instance dying. The chat service
# reaches it as PostgresSaver over a DSN it never sees in the image or the repo: the DSN is a
# Secret Manager secret, mounted as CHECKPOINT_DSN by `--set-secrets` (commands/lesson-12.8.sh).
#
# WHAT THIS COSTS. db-f1-micro, zonal, 10 GB, no backups: roughly $8-10 (Rs 700-850) a month
# while it exists - the cheapest Cloud SQL there is, and enough for checkpoints. `make down`
# destroys it; deletion_protection is off on purpose, because this is a learner's throwaway
# project and a destroy that stops at the database is a bill that keeps running.
#
# HOW IT IS REACHED. Public IP with NO authorized networks: nothing on the internet can open a
# connection. Cloud Run connects through the Cloud SQL connector (`--add-cloudsql-instances`),
# which authenticates the SERVICE ACCOUNT with IAM and exposes a unix socket at
# /cloudsql/PROJECT:REGION:INSTANCE - the `host=` in the DSN below, exactly as 8.5 wrote it.
# Private IP needs service-networking peering on documind-vpc; add it when 12.1's network
# lesson wants it, and nothing else here changes.

resource "random_password" "checkpoint" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  length  = 24
  special = false
}

resource "google_sql_database_instance" "checkpoint" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  name                = "documind-checkpoint"
  database_version    = "POSTGRES_16"
  region              = var.region
  deletion_protection = false

  settings {
    tier              = "db-f1-micro"
    availability_type = "ZONAL"
    disk_size         = 10
    ip_configuration {
      ipv4_enabled = true # reached only through the connector; no authorized_networks, on purpose
    }
    backup_configuration {
      enabled = false # checkpoints, not customer documents
    }
  }
}

resource "google_sql_database" "checkpoint" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  name     = "documind"
  instance = google_sql_database_instance.checkpoint[0].name
}

resource "google_sql_user" "chat" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  name     = "chat"
  instance = google_sql_database_instance.checkpoint[0].name
  password = random_password.checkpoint[0].result
}

# The DSN is the secret, in the shape 8.5 wrote: the socket directory rides in the query string.
resource "google_secret_manager_secret" "checkpoint_dsn" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  secret_id = "documind-checkpoint-dsn"
  replication {
    auto {}
  }
}

resource "google_secret_manager_secret_version" "checkpoint_dsn" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  secret      = google_secret_manager_secret.checkpoint_dsn[0].id
  secret_data = "postgresql://chat:${random_password.checkpoint[0].result}@/documind?host=/cloudsql/${google_sql_database_instance.checkpoint[0].connection_name}"
}

resource "google_secret_manager_secret_iam_member" "chat_dsn" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  secret_id = google_secret_manager_secret.checkpoint_dsn[0].id
  role      = "roles/secretmanager.secretAccessor"
  member    = "serviceAccount:${google_service_account.chat.email}"
}

# The connector authenticates the service account, and this is the role it checks.
resource "google_project_iam_member" "chat_sql_client" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  project = var.project_id
  role    = "roles/cloudsql.client"
  member  = "serviceAccount:${google_service_account.chat.email}"
}

output "checkpoint_instance" {
  description = "Pass to --add-cloudsql-instances on documind-chat and to the migration job"
  value       = one(google_sql_database_instance.checkpoint[*].connection_name)
}
'''

with open('cloudsql.tf', 'w') as f: f.write(CLOUDSQL_TF)
print('cloudsql.tf:', len(CLOUDSQL_TF.splitlines()), 'lines')


In [ ]:
DEPLOY = '''
# 1. The image, from the deploy/ context - the same config as rag-api, another Dockerfile.
gcloud builds submit --config=cloudbuild.yaml \\
  --substitutions=_IMAGE=us-central1-docker.pkg.dev/$PROJECT/documind/chat:$GIT_SHA,_DOCKERFILE=services/chat/Dockerfile .

# 2. The service. Two things differ by profile, and the Makefile fills both (deploy-services):
#    CHAT_SQL_FLAGS  full: --add-cloudsql-instances + --set-secrets mount the DSN Terraform wrote
#                    (cloudsql.tf). lean: there is no Cloud SQL and the pair is empty - set, and
#                    empty, which is why the expansion below is ${VAR-default} and not ${VAR:-default}.
#    CHAT_EXTRA_ENV  lean: |CHECKPOINT_DSN=memory - the in-memory checkpointer, which agent.py logs
#                    as "tests only" because a conversation dies with the instance (8.5). A demo can
#                    live with that; a product cannot, and the full profile says so with a database.
#    DOCUMIND_PROFILE=gcp is stated so agent.py's local bypass is unreachable. IAP_AUDIENCE lists
#    this surface AND the UI: the UI forwards the person's assertion when its brain radio calls
#    this service (12.4), and that assertion was minted for the UI's audience. SELF_URL is the
#    bearer leg: an agent, make smoke-chat or 8.7's notebook calls with an ID token minted for this
#    URL and no assertion, shared/iap.identity verifies it, and the roster still decides.
#    RAG_TIMEOUT_S=90 is 7.2's finding - a cold API takes longer than the tool layer's default.
#    DOCUMIND_BRAIN is the default brain; GOOGLE_GENAI_USE_VERTEXAI is for the ADK brain.
gcloud run deploy documind-chat \\
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/chat:$GIT_SHA \\
  --region=us-central1 --platform=managed \\
  --no-allow-unauthenticated \\
  --memory=1Gi --cpu=1 --concurrency=20 --timeout=300 \\
  --min-instances=0 --max-instances=10 \\
  --service-account=documind-chat-sa@$PROJECT.iam.gserviceaccount.com \\
  ${CHAT_SQL_FLAGS---add-cloudsql-instances=$PROJECT:us-central1:documind-checkpoint --set-secrets=CHECKPOINT_DSN=documind-checkpoint-dsn:latest} \\
  --set-env-vars="^|^GOOGLE_CLOUD_PROJECT=$PROJECT|DOCUMIND_PROFILE=gcp|RAG_API_URL=https://documind-api-$PROJECT_NUMBER.us-central1.run.app|SELF_URL=https://documind-chat-$PROJECT_NUMBER.us-central1.run.app|RAG_TIMEOUT_S=90|DOCUMIND_BRAIN=langchain|GOOGLE_GENAI_USE_VERTEXAI=1|GOOGLE_CLOUD_LOCATION=global|IAP_AUDIENCE=/projects/$PROJECT_NUMBER/locations/us-central1/services/documind-chat,/projects/$PROJECT_NUMBER/locations/us-central1/services/documind-ui${CHAT_EXTRA_ENV-}"

# 3. The one-time checkpoint migration (8.5: setup() takes exclusive locks - a job, never startup).
#    Full profile only: the lean lane has no database to migrate.
if [ "${PROFILE-full}" = full ]; then
gcloud run jobs create documind-checkpoint-setup \\
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/chat:$GIT_SHA \\
  --region=us-central1 --service-account=documind-chat-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-cloudsql-instances=$PROJECT:us-central1:documind-checkpoint \\
  --set-secrets=CHECKPOINT_DSN=documind-checkpoint-dsn:latest \\
  --command=python --args=migrate.py || echo "job exists - continuing"
gcloud run jobs execute documind-checkpoint-setup --region=us-central1 --wait

# 4. IAP in front of the human surface, AFTER the service exists (step 4 of the runbook above).
#    Full profile only: on lean this service is a backend the UI and the smoke test call with ID
#    tokens, and IAP would refuse exactly those callers.
gcloud beta run services update documind-chat --region=us-central1 --iap
fi
'''
print(DEPLOY)
